# Step 12 — FastAPI Service
**Tough Talks · Phase 5**

Goal: expose every Phase 1–4 intelligence component as a local HTTP endpoint, then drive every endpoint end-to-end through FastAPI's `TestClient` and confirm each response matches the JSON schema the component already emits. This is the first integration step in the project — every prior phase operated on one component at a time.

**Architecture decisions** (confirmed at planning time):

- **One endpoint per component** (no session orchestration in the backend). The API is stateless; storage is Step 13's job.
  - `POST /talk-dna/analyze` (text) — wraps `analyze_talk_dna`.
  - `POST /vault/build` (text) — wraps `analyze_person_vault`.
  - `POST /persona/reply` and `POST /persona/run` (text) — wrap `generate_persona_reply` / `run_practice_conversation`.
  - `POST /premortem` (text) — wraps `generate_premortem`.
  - `POST /debrief` (text) — wraps `generate_debrief`.
  - `POST /aftermath` (text) — wraps `generate_aftermath`.
  - `POST /pulse` (text) — wraps `generate_pulse`.
  - `POST /transcribe` (multimodal, multipart) — wraps `transcribe_long`.
  - `POST /emotion/analyze` (multimodal, multipart) — wraps `analyze_emotion_long`.
- **Eager model load at app startup** via FastAPI `lifespan`. Both Gemma 4 variants (text-only `AutoModelForCausalLM` and multimodal `AutoModelForMultimodalLM`) live on one `ModelRegistry` attached to `app.state.registry`. Notebook / test code skips the lifespan and injects a registry via `app.dependency_overrides[get_registry]` so the same loaded weights serve every route in this cell.
- **Multipart audio uploads** (`UploadFile`) on the two audio routes — the shape browser `MediaRecorder` + `FormData` will emit in Phase 6. The route streams the upload to a `NamedTemporaryFile` and passes the path to `transcribe_long` / `analyze_emotion_long`.
- **`TestClient` (in-process) for this notebook** — no uvicorn subprocess. Each endpoint hits real ASGI / Pydantic / dependency injection with the loaded models attached; only the network hop is skipped.

**What `done` looks like for this step**

1. Both Gemma 4 variants load and are attached to a `ModelRegistry`.
2. `GET /health` returns `{status: ok, text_model_loaded: True, multimodal_model_loaded: True}`.
3. Each text route returns 200 with a response matching its `data/schemas/*.schema.json` (structural checks: required keys present, enum values in range, list cardinality constraints honoured).
4. The two audio routes accept a small gTTS-generated WAV (MIT-licensed, in-notebook synthesis — no third-party audio download — stays Kaggle-license-safe per `[[project_kaggle_license_safety]]`), return 200, and produce schema-conforming output.
5. The final cell renders a single pass/fail table over all endpoints. The cell runs unconditionally so a failure on one endpoint doesn't hide failures on the others (`[[rule]] Final cell is always a results table that renders unconditionally`).

**Note on fixtures.** Jamie's PersonVault is the same v2 profile used across Steps 6–11. The practice transcript is the same Step 7 emit shape used by Steps 9 / 10. Reusing fixtures keeps the integration test focused on the API surface, not on regenerating the inputs each route consumes.

In [1]:
# ── 0. Install / upgrade dependencies ────────────────
# Same rule as Steps 05–11: bump only transformers + accelerate on
# Colab / Kaggle (bumping torch breaks the pre-installed torchvision /
# CUDA pairing). The FastAPI / Pydantic / multipart / httpx / soundfile
# / librosa / gTTS deps are all already pinned in requirements.txt —
# Colab / Kaggle pre-installs the first four but not gTTS, hence the
# explicit install.
#
# After this first run, RESTART THE KERNEL before continuing if you
# actually upgraded transformers — the already-imported version won't
# pick up the change.

!pip install -q -U transformers accelerate
!pip install -q fastapi 'pydantic>=2.6' python-multipart httpx soundfile librosa gTTS

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.5/110.5 kB 7.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gtts 2.5.4 requires click<8.2,>=7.1, but you have click 8.3.3 which is incompatible.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
typer 0.24.2 requires click>=8.2.1, but you have click 8.1.8 which is incompatible.


In [2]:
# ── 1. Locate (or fetch) the repo, put it on sys.path ───────────
# Same shim as Steps 01–11.

import os, pathlib, subprocess, sys

REPO_URL  = "https://github.com/EhsanFarazmand/tough_talks.git"
REPO_NAME = "tough_talks"

def _looks_like_repo(p: pathlib.Path) -> bool:
    return (p / "backend" / "core" / "_runtime").is_dir()

def _scan_for_repo() -> pathlib.Path | None:
    cwd = pathlib.Path.cwd()
    for parent in [cwd, *cwd.parents]:
        if _looks_like_repo(parent):
            return parent
    for base in (pathlib.Path("/content"), pathlib.Path("/kaggle/working")):
        candidate = base / REPO_NAME
        if _looks_like_repo(candidate):
            return candidate
    return None

def _refresh(target: pathlib.Path) -> None:
    if not (target / ".git").is_dir():
        return
    print(f"Refreshing {target} from origin")
    subprocess.run(["git", "-C", str(target), "fetch", "--depth", "1", "origin"],
                   capture_output=True, check=False)
    subprocess.run(["git", "-C", str(target), "reset", "--hard", "FETCH_HEAD"],
                   capture_output=True, check=False)

REPO_ROOT = _scan_for_repo()
if REPO_ROOT is None:
    base = next((b for b in (pathlib.Path("/content"), pathlib.Path("/kaggle/working")) if b.is_dir()),
                pathlib.Path.cwd())
    target = base / REPO_NAME
    print(f"Cloning {REPO_URL} -> {target}")
    result = subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(target)],
                            capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError("git clone failed:\n" + result.stderr)
    REPO_ROOT = target
else:
    _refresh(REPO_ROOT)

os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

_stale = [m for m in list(sys.modules) if m == "backend" or m.startswith("backend.")]
for _m in _stale:
    del sys.modules[_m]
if _stale:
    print(f"Cleared {len(_stale)} cached backend.* module(s) from sys.modules")

print(f"Repo root: {REPO_ROOT}")

Refreshing /content/tough_talks from origin
Repo root: /content/tough_talks


In [3]:
# ── 2. Imports ──────────────────────────────
import io
import json
import tempfile
from pathlib import Path

import torch
from fastapi.testclient import TestClient

from backend.api.deps import ModelRegistry, get_registry
from backend.api.main import app
from backend.core._runtime import (
    ALLOWED_EMOTIONS,
    ALLOWED_HEALTH_TRENDS,
    ALLOWED_MATCH_QUALITIES,
    ALLOWED_RESISTANCE_TYPES,
    DEFAULT_MODEL_ID,
    LoadConfig,
    load_model,
)

In [4]:
# ── 3. Config ───────────────────────────────
MODEL_ID = DEFAULT_MODEL_ID  # google/gemma-4-E2B-it

print("Model              :", MODEL_ID)
print("CUDA available     :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA device        :", torch.cuda.get_device_name(0))
    print("Free VRAM (GB)     :", round(torch.cuda.mem_get_info()[0] / 1e9, 2))
print("Repo root          :", REPO_ROOT)
print("Schemas dir        :", REPO_ROOT / "data" / "schemas")

Model              : google/gemma-4-E2B-it
CUDA available     : True
CUDA device        : Tesla T4
Free VRAM (GB)     : 15.53
Repo root          : /content/tough_talks
Schemas dir        : /content/tough_talks/data/schemas


In [5]:
# ── 4. Load the multimodal Gemma 4 variant once ────────────────
# Single-model strategy: load ONLY the multimodal variant and use it
# for every route — text-only and audio alike. The multimodal class
# wraps the same LM as the text-only class; text generation works on
# either, and the extra audio/image projection heads on the multimodal
# variant simply go unused on a text-only chat.
#
# Why we don't load both: on a T4 (15.5 GB VRAM) the text-only model
# eats ~10 GB; the multimodal model then can't fit and `accelerate`
# silently offloads some parameters to the meta device. Inference
# blows up later with `Tensor on device meta is not on the expected
# device cuda:0`. Loading one variant keeps ~5 GB of VRAM free for
# the KV cache and matches the eventual on-device deployment target.
#
# `ModelRegistry.text()` falls back to the multimodal pair when the
# dedicated text-only pair is missing — so this exact registry shape
# serves every route in the notebook.

print("Loading multimodal model ...")
mm_processor, mm_model = load_model(LoadConfig(model_id=MODEL_ID, multimodal=True))
print("  multimodal device     :", mm_model.device)
print("  multimodal dtype      :", next(mm_model.parameters()).dtype)

if torch.cuda.is_available():
    print("Remaining free VRAM   :", round(torch.cuda.mem_get_info()[0] / 1e9, 2), "GB")

Loading multimodal model ...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

  multimodal device     : cuda:0
  multimodal dtype      : torch.bfloat16
Remaining free VRAM   : 5.13 GB


In [6]:
# ── 5. Build a ModelRegistry + TestClient ───────────────────
# Skip the lifespan event entirely (no `with` block on TestClient).
# Inject a registry holding the notebook-loaded multimodal pair via
# app.dependency_overrides — every route gets the same weights without
# a second load. Text routes go through the registry's text() fallback
# to the multimodal pair (see backend/api/deps.py:ModelRegistry.text).

registry = ModelRegistry(
    multimodal_processor=mm_processor,
    multimodal_model=mm_model,
)
# /health reads directly from app.state, so the smoke check below sees
# the same registry the routes do. text_model_loaded will be False
# (we didn't load a dedicated text pair); multimodal_model_loaded will
# be True — exactly the production shape we're targeting on T4.
app.state.registry = registry
app.dependency_overrides[get_registry] = lambda: registry

client = TestClient(app)
print("TestClient ready.")
print("Routes exposed:")
for r in sorted(app.routes, key=lambda r: getattr(r, 'path', '')):
    methods = ','.join(sorted(getattr(r, 'methods', set()) - {'HEAD'}))
    print(f"  {methods:8s} {getattr(r, 'path', '?')}")

TestClient ready.
Routes exposed:
  POST     /aftermath
  POST     /debrief
  GET      /docs
  GET      /docs/oauth2-redirect
  POST     /emotion/analyze
  GET      /health
  GET      /openapi.json
  POST     /persona/reply
  POST     /persona/run
  POST     /premortem
  POST     /pulse
  GET      /redoc
  POST     /talk-dna/analyze
  POST     /transcribe
  POST     /vault/build


In [7]:
# ── 6. Fixtures (Jamie + practice transcript) ─────────────────
# Same Jamie v2 PersonVault profile used across Steps 6–11.
JAMIE_VAULT = {
    "person_id": "person_cf528d57b789",
    "name": "Jamie",
    "relationship_type": "colleague",
    "version": 2,
    "conversation_count": 2,
    "profile": {
        "communication_style": "defensive",
        "emotional_triggers": [
            "citing past commitments",
            "hard deadlines without naming blockers",
        ],
        "de_escalation_keys": [
            "explicitly disowning blame",
            "reframing as joint problem-solving",
        ],
        "common_deflections": [
            "I told you the staging tables weren't done",
            "don't put this on me",
        ],
        "responds_best_to": "concrete next steps and shared ownership.",
    },
    "updated_at": "2026-05-14T15:00:00+00:00",
}

# Practice transcript shape from Step 07 — user + persona alternating.
PRACTICE_TRANSCRIPT = [
    {"speaker": "user", "text": "I noticed Tuesday's deadline slipped — what's going on?"},
    {
        "speaker": "persona",
        "persona_name": "Jamie",
        "reply": "I told you the staging tables weren't done last Friday.",
        "resistance_type": "deflect",
        "escalation_level": 0.55,
    },
    {"speaker": "user", "text": "I hear that. I'm not blaming you — I want to land Wednesday EOD together. What's the blocker we can name today?"},
    {
        "speaker": "persona",
        "persona_name": "Jamie",
        "reply": "The data team hasn't confirmed schema parity. If we agree on a daily Slack check-in I can commit to Wednesday.",
        "resistance_type": "concede",
        "escalation_level": 0.25,
    },
]

PRACTICE_USER_MESSAGES = [t["text"] for t in PRACTICE_TRANSCRIPT if t["speaker"] == "user"]
USER_GOAL = "Get Jamie to commit to Wednesday EOD without re-litigating the missed Tuesday deadline."

print(f"Loaded fixtures: Jamie vault, transcript ({len(PRACTICE_TRANSCRIPT)} turns), "
      f"{len(PRACTICE_USER_MESSAGES)} user messages.")

Loaded fixtures: Jamie vault, transcript (4 turns), 2 user messages.


In [ ]:
# ── 7. Hit every text route via TestClient ───────────────────
# Each block records (endpoint, ok, detail) into RESULTS so the final
# summary table renders unconditionally even if a route fails.
# Structural assertions only — the runtime coercers already enforce
# schema conformance; we just verify the wire shape survived FastAPI.

RESULTS: list[dict] = []

def _record(name: str, ok: bool, detail: str) -> None:
    RESULTS.append({"endpoint": name, "ok": ok, "detail": detail})
    print(f"{'PASS' if ok else 'FAIL'}  {name:30s} {detail}")

# --- /health -------------------------------------------------------------
# Single-model strategy: only the multimodal variant is loaded. The
# registry's text() falls back to the multimodal pair so text routes
# still work; text_model_loaded is honestly False in /health.
try:
    r = client.get("/health")
    assert r.status_code == 200, r.text
    body = r.json()
    assert body["status"] == "ok"
    assert body["multimodal_model_loaded"] is True
    _record("GET /health", True, (
        f"text_loaded={body['text_model_loaded']}, "
        f"multimodal_loaded={body['multimodal_model_loaded']}"
    ))
except Exception as exc:  # noqa: BLE001
    _record("GET /health", False, f"{type(exc).__name__}: {exc}")

# --- /talk-dna/analyze ---------------------------------------------------
try:
    r = client.post("/talk-dna/analyze", json={
        "turns": PRACTICE_TRANSCRIPT,
        "user_id": "local",
        "max_new_tokens": 512,
    })
    assert r.status_code == 200, r.text
    body = r.json()
    assert body["user_id"] == "local"
    assert isinstance(body["version"], int) and body["version"] >= 1
    patterns = body["patterns"]
    assert 0.0 <= patterns["apology_rate"] <= 1.0
    assert patterns["sarcasm_frequency"] in {"never", "rare", "low", "moderate", "high"}
    TALK_DNA_RESULT = body
    _record("POST /talk-dna/analyze", True, f"v{body['version']}, apology_rate={patterns['apology_rate']:.2f}")
except Exception as exc:  # noqa: BLE001
    TALK_DNA_RESULT = None
    _record("POST /talk-dna/analyze", False, f"{type(exc).__name__}: {exc}")

# --- /vault/build --------------------------------------------------------
# PersonVault feeds from real recorded conversations: it expects
# ``speaker == "other"`` with the counterparty's spoken content in
# ``text`` — the shape Step 03 transcription emits. The practice
# transcript above uses Step 07's emit shape (``persona`` + ``reply``)
# because debrief / aftermath / pulse all consume that shape. In
# production the frontend would hand each route the input matching its
# consumer; here we normalise inline so one fixture drives both data
# flows.
vault_turns = [
    {"speaker": "other", "text": t["reply"]} if t["speaker"] == "persona" else t
    for t in PRACTICE_TRANSCRIPT
]
try:
    r = client.post("/vault/build", json={
        "turns": vault_turns,
        "name": "Jamie",
        "relationship_type": "colleague",
        "max_new_tokens": 512,
    })
    assert r.status_code == 200, r.text
    body = r.json()
    assert body["name"] == "Jamie"
    assert body["relationship_type"] == "colleague"
    assert body["profile"]["communication_style"] in {
        "direct", "indirect", "passive_aggressive", "avoidant", "collaborative",
        "dominant", "assertive", "empathetic", "defensive",
    }
    VAULT_RESULT = body
    _record("POST /vault/build", True, f"style={body['profile']['communication_style']}")
except Exception as exc:  # noqa: BLE001
    VAULT_RESULT = None
    _record("POST /vault/build", False, f"{type(exc).__name__}: {exc}")

# --- /persona/reply ------------------------------------------------------
try:
    r = client.post("/persona/reply", json={
        "user_message": PRACTICE_USER_MESSAGES[0],
        "persona_profile": JAMIE_VAULT,
        "history": [],
        "user_goal": USER_GOAL,
        "max_new_tokens": 256,
    })
    assert r.status_code == 200, r.text
    body = r.json()
    assert body["persona_name"] == "Jamie"
    assert body["reply"].strip()
    assert body["resistance_type"] in ALLOWED_RESISTANCE_TYPES
    assert 0.0 <= body["escalation_level"] <= 1.0
    _record("POST /persona/reply", True, f"{body['resistance_type']} @ {body['escalation_level']:.2f}")
except Exception as exc:  # noqa: BLE001
    _record("POST /persona/reply", False, f"{type(exc).__name__}: {exc}")

# --- /persona/run --------------------------------------------------------
# Only the first user message — keeps the cell under one minute on E2B.
# A full multi-turn run is exercised by the dedicated Step 07 notebook.
try:
    r = client.post("/persona/run", json={
        "user_messages": PRACTICE_USER_MESSAGES[:1],
        "persona_profile": JAMIE_VAULT,
        "user_goal": USER_GOAL,
        "max_new_tokens": 256,
    })
    assert r.status_code == 200, r.text
    body = r.json()
    history = body["history"]
    assert len(history) == 2, history  # one user + one persona
    assert history[0]["speaker"] == "user"
    assert history[1]["speaker"] == "persona"
    assert history[1]["resistance_type"] in ALLOWED_RESISTANCE_TYPES
    _record("POST /persona/run", True, f"{len(history)} history turns")
except Exception as exc:  # noqa: BLE001
    _record("POST /persona/run", False, f"{type(exc).__name__}: {exc}")

# --- /premortem ----------------------------------------------------------
try:
    r = client.post("/premortem", json={
        "conversation_description": (
            "Tomorrow's 1-on-1 with Jamie about why Tuesday's data-pipeline "
            "deadline slipped again and re-committing to Wednesday EOD."
        ),
        "user_goal": USER_GOAL,
        "person_profile": JAMIE_VAULT,
        "talk_dna_profile": TALK_DNA_RESULT or {},
        "enable_thinking": True,
        "max_new_tokens": 2048,
    })
    assert r.status_code == 200, r.text
    body = r.json()
    scenarios = body["failure_scenarios"]
    assert len(scenarios) == 3, len(scenarios)
    for i, s in enumerate(scenarios, start=1):
        assert s["scenario_id"] == i
        assert s["simulation_parameters"]["resistance_type"] in ALLOWED_RESISTANCE_TYPES
        assert 0.0 <= s["destabilization_risk"] <= 1.0
        assert 0.0 <= s["simulation_parameters"]["escalation_ceiling"] <= 1.0
    PREMORTEM_RESULT = body
    _record("POST /premortem", True, f"3 scenarios, types=[{', '.join(s['simulation_parameters']['resistance_type'] for s in scenarios)}]")
except Exception as exc:  # noqa: BLE001
    PREMORTEM_RESULT = None
    _record("POST /premortem", False, f"{type(exc).__name__}: {exc}")

# --- /debrief ------------------------------------------------------------
try:
    r = client.post("/debrief", json={
        "transcript": PRACTICE_TRANSCRIPT,
        "user_goal": USER_GOAL,
        "person_profile": JAMIE_VAULT,
        "talk_dna_profile": TALK_DNA_RESULT or {},
        "enable_thinking": True,
        "max_new_tokens": 2048,
    })
    assert r.status_code == 200, r.text
    body = r.json()
    assert isinstance(body["ground_lost"], list)
    assert isinstance(body["wins"], list)
    assert isinstance(body["over_apologies"], list)
    assert isinstance(body["missed_openings"], list)
    assert body["one_fix_next_time"].strip()
    DEBRIEF_RESULT = body
    _record("POST /debrief", True, (
        f"wins={len(body['wins'])}, ground_lost={len(body['ground_lost'])}, "
        f"over_apologies={len(body['over_apologies'])}, missed={len(body['missed_openings'])}"
    ))
except Exception as exc:  # noqa: BLE001
    DEBRIEF_RESULT = None
    _record("POST /debrief", False, f"{type(exc).__name__}: {exc}")

# --- /aftermath ----------------------------------------------------------
try:
    if PREMORTEM_RESULT is None:
        raise RuntimeError("skipped: /premortem did not produce a payload")
    r = client.post("/aftermath", json={
        "premortem": PREMORTEM_RESULT,
        "transcript": PRACTICE_TRANSCRIPT,
        "user_goal": USER_GOAL,
        "person_profile": JAMIE_VAULT,
        "debrief": DEBRIEF_RESULT or {},
        "enable_thinking": True,
        "max_new_tokens": 4096,
    })
    assert r.status_code == 200, r.text
    body = r.json()
    assert body["goal_outcome"]["status"] in {"achieved", "partial", "not_achieved"}
    assert len(body["scenario_outcomes"]) == 3
    for i, outcome in enumerate(body["scenario_outcomes"], start=1):
        assert outcome["scenario_id"] == i
        assert outcome["match_quality"] in ALLOWED_MATCH_QUALITIES
        # materialized derived from match_quality — see runtime contract.
        assert outcome["materialized"] is (outcome["match_quality"] != "did_not_occur")
    assert 0.0 <= body["prediction_accuracy"] <= 1.0
    AFTERMATH_RESULT = body
    _record("POST /aftermath", True, (
        f"{body['goal_outcome']['status']}, accuracy={body['prediction_accuracy']:.2f}"
    ))
except Exception as exc:  # noqa: BLE001
    AFTERMATH_RESULT = None
    _record("POST /aftermath", False, f"{type(exc).__name__}: {exc}")

# --- /pulse --------------------------------------------------------------
try:
    # Use two synthetic rounds with the same aftermath + debrief surrogate
    # so the pulse has a real two-round trajectory to roll up.
    rounds = [
        {
            "round_id": "round_a",
            "started_at": "2026-04-30T16:00:00+00:00",
            "user_goal": USER_GOAL,
            "aftermath": AFTERMATH_RESULT or {},
            "debrief": DEBRIEF_RESULT or {},
        },
        {
            "round_id": "round_b",
            "started_at": "2026-05-14T15:30:00+00:00",
            "user_goal": USER_GOAL,
            "aftermath": AFTERMATH_RESULT or {},
            "debrief": DEBRIEF_RESULT or {},
        },
    ]
    r = client.post("/pulse", json={
        "rounds": rounds,
        "person_profile": JAMIE_VAULT,
        "enable_thinking": True,
        "max_new_tokens": 4096,
    })
    assert r.status_code == 200, r.text
    body = r.json()
    assert body["person_id"] == JAMIE_VAULT["person_id"]
    assert body["version"] == 1
    assert body["round_count"] == 2
    assert body["health_trend"] in ALLOWED_HEALTH_TRENDS
    assert 0.0 <= body["health_score"] <= 1.0
    assert len(body["round_summaries"]) == 2
    # round_id / started_at must be carried through verbatim from input.
    assert body["round_summaries"][0]["round_id"] == "round_a"
    assert body["round_summaries"][1]["round_id"] == "round_b"
    # Every recurring pattern must cite ≥ 2 known round_ids.
    known_ids = {s["round_id"] for s in body["round_summaries"]}
    for pat in body["recurring_patterns"]:
        assert len(set(pat["evidence_round_ids"]) & known_ids) >= 2
    _record("POST /pulse", True, (
        f"v{body['version']}, trend={body['health_trend']}, score={body['health_score']:.2f}, "
        f"patterns={len(body['recurring_patterns'])}, wins={len(body['relationship_wins'])}"
    ))
except Exception as exc:  # noqa: BLE001
    _record("POST /pulse", False, f"{type(exc).__name__}: {exc}")

In [9]:
# ── 8. Hit the two audio routes ──────────────────────────
# Synthesise a short clip with gTTS (MIT-licensed, in-notebook — keeps
# the Kaggle submission license-clean per `[[project_kaggle_license_safety]]`).
# gTTS produces MP3; we re-encode to mono-16kHz WAV via librosa /
# soundfile so the path matches what Gemma 4's audio feature extractor
# expects.

import librosa
import soundfile as sf
from gtts import gTTS

TTS_TEXT = "This is a short test clip for the Tough Talks API."

audio_dir = Path(tempfile.mkdtemp(prefix="step12_audio_"))
mp3_path = audio_dir / "clip.mp3"
wav_path = audio_dir / "clip.wav"

tts = gTTS(text=TTS_TEXT, lang="en")
tts.save(str(mp3_path))
wave, sr = librosa.load(str(mp3_path), sr=16000, mono=True)
sf.write(str(wav_path), wave, 16000)
audio_bytes = wav_path.read_bytes()
print(f"Test audio: {wav_path.name} ({len(audio_bytes)} bytes, {len(wave)/16000:.2f} s)")

# --- /transcribe ---------------------------------------------------------
try:
    r = client.post(
        "/transcribe",
        files={"audio": ("clip.wav", audio_bytes, "audio/wav")},
        data={"max_new_tokens": "128", "language": "en"},
    )
    assert r.status_code == 200, r.text
    body = r.json()
    assert body["transcript"].strip(), body
    assert body["language"] == "en"
    _record("POST /transcribe", True, f"transcript={body['transcript']!r}")
except Exception as exc:  # noqa: BLE001
    _record("POST /transcribe", False, f"{type(exc).__name__}: {exc}")

# --- /emotion/analyze ----------------------------------------------------
try:
    r = client.post(
        "/emotion/analyze",
        files={"audio": ("clip.wav", audio_bytes, "audio/wav")},
        data={
            "speaker": "user",
            "turn_id": "step12_t1",
            "transcript_snippet": TTS_TEXT,
            "max_new_tokens": "256",
        },
    )
    assert r.status_code == 200, r.text
    body = r.json()
    results = body["results"]
    assert len(results) >= 1
    first = results[0]
    assert first["speaker"] == "user"
    emotions = first["emotions"]
    assert emotions["primary"] in ALLOWED_EMOTIONS
    for k in ("intensity", "tension_level", "escalation_risk"):
        assert 0.0 <= emotions[k] <= 1.0
    _record("POST /emotion/analyze", True, (
        f"{len(results)} chunk(s), primary={emotions['primary']}, intensity={emotions['intensity']:.2f}"
    ))
except Exception as exc:  # noqa: BLE001
    _record("POST /emotion/analyze", False, f"{type(exc).__name__}: {exc}")

Test audio: clip.wav (135980 bytes, 4.25 s)
PASS  POST /transcribe               transcript='This is a short test clip for the Tough Talks API.'
PASS  POST /emotion/analyze          1 chunk(s), primary=defensiveness, intensity=0.70


In [10]:
# ── 9. Final summary table (renders unconditionally) ─────────────
# One row per endpoint, pass/fail + a one-line detail. This cell
# never asserts — a single bad route should not hide the others. The
# pass/fail decision per row was already recorded above; this just
# renders them.

if not RESULTS:
    print("No results recorded — the earlier cells did not run.")
else:
    name_w = max(len(r["endpoint"]) for r in RESULTS)
    header = f"  {'endpoint':{name_w}s}  status  detail"
    rule = "  " + "-" * (name_w + 8) + "-" * 60
    print(header)
    print(rule)
    for row in RESULTS:
        mark = "PASS" if row["ok"] else "FAIL"
        print(f"  {row['endpoint']:{name_w}s}  {mark:6s}  {row['detail']}")
    print(rule)
    passed = sum(1 for r in RESULTS if r["ok"])
    total = len(RESULTS)
    print(f"  {passed}/{total} endpoints OK")

  endpoint                status  detail
  ------------------------------------------------------------------------------------------
  GET /health             PASS    text_loaded=False, multimodal_loaded=True
  POST /talk-dna/analyze  PASS    v1, apology_rate=0.00
  POST /vault/build       PASS    style=assertive
  POST /persona/reply     PASS    deflect @ 0.60
  POST /persona/run       PASS    2 history turns
  POST /premortem         PASS    3 scenarios, types=[deflect, guilt_trip, deny]
  POST /debrief           PASS    wins=1, ground_lost=0, over_apologies=0, missed=0
  POST /aftermath         PASS    partial, accuracy=0.65
  POST /pulse             PASS    v1, trend=stable, score=0.65, patterns=1, wins=1
  POST /transcribe        PASS    transcript='This is a short test clip for the Tough Talks API.'
  POST /emotion/analyze   PASS    1 chunk(s), primary=defensiveness, intensity=0.70
  ------------------------------------------------------------------------------------------
  11/